# DimASR: Dimensional Aspect Sentiment Regression - Laptop Domain

**SemEval-2026 Task 3 - Track A: Subtask 1**

This notebook implements the DimASR task which predicts valence-arousal (VA) scores for given aspects in review text.
- **Valence**: Emotional positivity (1.00=very negative, 5.00=neutral, 9.00=very positive)
- **Arousal**: Emotional intensity (1.00=calm, 9.00=excited/intense)

Output format: `valence#arousal` (e.g., "7.12#6.88")

## Libraries

In [ ]:
try:
    import google.colab
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    IN_COLAB = True
except:
    IN_COLAB = False

In [ ]:
if IN_COLAB:
    !pip install transformers datasets evaluate sentencepiece scipy

In [ ]:
import os
import json
import torch
import warnings
import pandas as pd
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')

if IN_COLAB:
    root_path = 'Enter drive path'
else:
    root_path = '/mnt/f/Jyothi/ITATA'

use_mps = True if torch.backends.mps.is_built() else False
os.chdir(root_path)
print(f"Working directory: {os.getcwd()}")
print(f"MPS available: {use_mps}")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
from Imports.data_prep import DatasetLoader
from Imports.utils import T5Generator
from instructions import InstructionsHandler

## Configuration

In [ ]:
# Task configuration
task_name = 'dimasr'
experiment_name = 'laptop_eng_v1'
model_checkpoint = 'allenai/tk-instruct-base-def-pos'  # or 'google/flan-t5-base'

# Data paths
train_file = './eng_laptop_train_alltasks.jsonl'
dev_file = './eng_laptop_dev_task1.jsonl'

# Model output path
model_out_path = os.path.join('./Models', task_name, f"{model_checkpoint.replace('/', '')}-{experiment_name}")
print('Experiment Name:', experiment_name)
print('Model output path:', model_out_path)

## Load Data

In [ ]:
# Load JSONL training data
train_df = DatasetLoader.load_jsonl_data(train_file)
dev_df = DatasetLoader.load_jsonl_data(dev_file)

print(f"Training records: {len(train_df)}")
print(f"Dev records: {len(dev_df)}")
print(f"\nTraining columns: {train_df.columns.tolist()}")
print(f"Dev columns: {dev_df.columns.tolist()}")

In [ ]:
# Display sample data
print("Sample training record:")
print(train_df.iloc[0])

## Setup Instructions

In [ ]:
# Initialize instruction handler
instruct_handler = InstructionsHandler()

# Load instruction set (use set1 for basic, set2 for extended examples, set3 for most examples)
instruct_handler.load_instruction_set2()

print("DimASR Instructions loaded:")
print(f"- BOS instruction length: {len(instruct_handler.dimasr['bos_instruct1'])} chars")
print(f"- Delimiter: '{instruct_handler.dimasr['delim_instruct']}'")
print(f"- EOS: '{instruct_handler.dimasr['eos_instruct']}'")

## Format Training Data

In [ ]:
# Create data loader and format training data
loader = DatasetLoader(train_df_id=None, test_df_id=None)

# Format training data (expand Quadruplet to individual rows)
train_formatted = loader.create_data_in_dimasr_format(
    train_df,
    text_col='Text',
    quadruplet_col='Quadruplet',
    bos_instruction=instruct_handler.dimasr['bos_instruct1'],  # bos_instruct1 for laptop domain
    delim_instruction=instruct_handler.dimasr['delim_instruct'],
    eos_instruction=instruct_handler.dimasr['eos_instruct'],
    is_train=True
)

print(f"Expanded training samples: {len(train_formatted)}")
print(f"\nColumns: {train_formatted.columns.tolist()}")

In [ ]:
# Display sample formatted data
print("Sample formatted input (first 500 chars):")
print(train_formatted['text'].iloc[0][:500])
print("\n...")
print(f"\nLabel (VA): {train_formatted['labels'].iloc[0]}")
print(f"Aspect: {train_formatted['aspect'].iloc[0]}")

## Train/Validation Split

In [ ]:
# Split into training and validation sets
train_split, val_split = train_test_split(train_formatted, test_size=0.1, random_state=42)

# Update loader with split data
loader.train_df_id = train_split.reset_index(drop=True)
loader.val_df_id = val_split.reset_index(drop=True)

print(f"Training samples: {len(train_split)}")
print(f"Validation samples: {len(val_split)}")

## Initialize Model

In [ ]:
# Create T5 Generator
t5_exp = T5Generator(model_checkpoint)
print(f"Model loaded: {model_checkpoint}")
print(f"Device: {t5_exp.device}")

## Tokenize Dataset

In [ ]:
# Tokenize datasets
id_ds, id_tokenized_ds, ood_ds, ood_tokenized_ds = loader.set_data_for_training_semeval(
    t5_exp.tokenize_function_inputs
)

print(f"Tokenized train samples: {len(id_tokenized_ds['train'])}")
print(f"Tokenized validation samples: {len(id_tokenized_ds['validation'])}")

## Training

In [ ]:
# Training arguments
training_args = {
    'output_dir': model_out_path,
    'evaluation_strategy': 'epoch',
    'learning_rate': 5e-5,
    'lr_scheduler_type': 'cosine',
    'per_device_train_batch_size': 8,
    'per_device_eval_batch_size': 16,
    'num_train_epochs': 5,
    'weight_decay': 0.01,
    'warmup_ratio': 0.1,
    'save_strategy': 'epoch',
    'load_best_model_at_end': True,
    'metric_for_best_model': 'eval_loss',
    'greater_is_better': False,
    'push_to_hub': False,
    'eval_accumulation_steps': 1,
    'predict_with_generate': True,
    'use_mps_device': use_mps
}

print("Training configuration:")
for k, v in training_args.items():
    print(f"  {k}: {v}")

In [ ]:
# Train the model
model_trainer = t5_exp.train(id_tokenized_ds, **training_args)

## Inference & Evaluation

In [ ]:
# Load trained model for inference
t5_exp = T5Generator(model_out_path)
print(f"Loaded trained model from: {model_out_path}")

In [ ]:
# Re-tokenize for inference
id_ds, id_tokenized_ds, _, _ = loader.set_data_for_training_semeval(t5_exp.tokenize_function_inputs)

# Get predictions on validation set
val_pred = t5_exp.get_labels(id_tokenized_ds, sample_set='validation', batch_size=16)
val_true = [label.strip() for label in id_ds['validation']['labels']]

print(f"Generated {len(val_pred)} predictions")

In [ ]:
# Calculate metrics
metrics = t5_exp.get_metrics_regression(val_true, val_pred)

print("\n" + "="*50)
print("VALIDATION METRICS")
print("="*50)
print(f"RMSE_VA (Official): {metrics['RMSE_VA']:.4f}")
print(f"PCC Valence:        {metrics['PCC_V']:.4f}")
print(f"PCC Arousal:        {metrics['PCC_A']:.4f}")
print(f"PCC Average:        {metrics['PCC_avg']:.4f}")
print(f"RMSE Valence:       {metrics['RMSE_V']:.4f}")
print(f"RMSE Arousal:       {metrics['RMSE_A']:.4f}")
print("="*50)

In [ ]:
# Display sample predictions
print("\nSample Predictions vs Ground Truth:")
print("-" * 60)
for i in range(min(10, len(val_pred))):
    pred_formatted = t5_exp.parse_va_prediction(val_pred[i])
    print(f"True: {val_true[i]:12} | Pred: {val_pred[i]:12} | Formatted: {pred_formatted}")

## Generate Predictions for Dev Set (Submission)

In [ ]:
# Format dev data for inference (no labels)
dev_formatted = loader.create_data_in_dimasr_format(
    dev_df,
    text_col='Text',
    aspect_col='Aspect',
    bos_instruction=instruct_handler.dimasr['bos_instruct1'],
    delim_instruction=instruct_handler.dimasr['delim_instruct'],
    eos_instruction=instruct_handler.dimasr['eos_instruct'],
    is_train=False
)

print(f"Dev samples to predict: {len(dev_formatted)}")

In [ ]:
# Create loader for dev predictions
loader_dev = DatasetLoader(train_df_id=dev_formatted, test_df_id=None)
dev_ds, dev_tokenized_ds, _, _ = loader_dev.set_data_for_training_semeval(t5_exp.tokenize_function_inputs)

# Get predictions
dev_predictions = t5_exp.get_labels(dev_tokenized_ds, sample_set='train', batch_size=16)
print(f"Generated {len(dev_predictions)} predictions for dev set")

In [ ]:
# Format output as required JSONL
def format_output_jsonl(dev_formatted_df, predictions, output_path, t5_model):
    """
    Format predictions into required output JSONL format.
    
    Output format:
    {"ID": "...", "Aspect_VA": [{"Aspect": "...", "VA": "X.XX#X.XX"}, ...]}
    """
    # Group predictions by ID
    results = {}
    
    for idx, row in dev_formatted_df.iterrows():
        record_id = row['ID']
        aspect = row['aspect']
        va_pred = t5_model.parse_va_prediction(predictions[idx])
        
        if record_id not in results:
            results[record_id] = []
        
        results[record_id].append({
            'Aspect': aspect,
            'VA': va_pred
        })
    
    # Write output JSONL
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with open(output_path, 'w', encoding='utf-8') as f:
        for record_id, aspect_vas in results.items():
            output_record = {
                'ID': record_id,
                'Aspect_VA': aspect_vas
            }
            f.write(json.dumps(output_record, ensure_ascii=False) + '\n')
    
    print(f"Output saved to: {output_path}")
    return results

# Generate output file
output_path = './predictions/pred_eng_laptop.jsonl'
results = format_output_jsonl(dev_formatted, dev_predictions, output_path, t5_exp)

In [ ]:
# Display sample predictions
print("\nSample predictions for submission:")
print("-" * 60)
for i, (record_id, aspect_vas) in enumerate(list(results.items())[:5]):
    print(f"\nID: {record_id}")
    for av in aspect_vas:
        print(f"  Aspect: {av['Aspect']:20} VA: {av['VA']}")

In [ ]:
# Verify output file format
print("\nFirst 3 lines of output file:")
print("-" * 60)
with open(output_path, 'r') as f:
    for i, line in enumerate(f):
        if i >= 3:
            break
        print(line.strip())

## Summary

This notebook has:
1. Loaded the DimASR training data (JSONL format with Quadruplet annotations)
2. Formatted the data with instruction prompts for the T5 model
3. Trained the model to generate valence#arousal scores
4. Evaluated on validation set using official RMSE_VA metric
5. Generated predictions for the dev set in submission format

The output file `predictions/pred_eng_laptop.jsonl` is ready for submission.